<!--
Copyright (c) 2025-2026, NVIDIA CORPORATION. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

 http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Aim
The purpose of this notebook is to show how to use JIT to tune a simple torch model.

In [1]:
%%capture
%load_ext autoreload
%autoreload 2
%cd ../..

In [ ]:
from aitune.torch.jit.patcher import patch_for_jit_tuning
from aitune.torch.jit.patched_module import PatchedModule
from aitune.torch.jit.config import config
import torch.nn as nn
import torch

Unable to import quantization op. Please install modelopt library (https://github.com/NVIDIA/TensorRT-Model-Optimizer?tab=readme-ov-file#installation) to add support for compiling quantized models
TensorRT-LLM is not installed. Please install TensorRT-LLM or set TRTLLM_PLUGINS_PATH to the directory containing libnvinfer_plugin_tensorrt_llm.so to use converters for torch.distributed ops


[08/05/2025-12:37:18] [TRT] [W] Functionality provided through tensorrt.plugin module is experimental.


# Simple model

In [3]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

The only requirement is to decorate model creation.

In [ ]:
@patch_for_jit_tuning
def create_model():
    return SimpleNet(10, 5, 2).to("cuda")


In [5]:
model = create_model()

At the beginning jit will try to detect model architecture. After two samples it will try to tune it.

It will also look at most at 3 levels depth of a model.


In [6]:
config.dry_run = False
config

Config(dry_run=False, dry_run_failure_probability=0.2, min_samples=2, batch_axis_required=True, max_depth_level=3, detect_graph_breaks=True, skip_modules=[], cache_dir=PosixPath('/home/pbazan/.cache/aitune.jit'))

In [7]:
model(torch.randn(2, 10, device="cuda"))

tensor([[-0.4038,  0.2994],
        [-0.4175,  0.3963]], device='cuda:0', grad_fn=<AddmmBackward0>)

Let's see detected model architecture.

In [8]:
PatchedModule.print_hierarchy()

PatchedModule Hierarchy:
├─ SimpleNet 📊97 level=0🪜 state=recording🔴 call_count=1
  ├─ Linear 📊55 level=1🪜 state=recording🔴 call_count=1
  ├─ Linear 📊30 level=1🪜 state=recording🔴 call_count=1
  ├─ Linear 📊12 level=1🪜 state=recording🔴 call_count=1


Now at 2nd call there will be tuning phase.

In [9]:
model(torch.randn(3, 10, device="cuda"))

  warnings.warn(

  warnings.warn(

  warnings.warn(



[W] 'colored' module is not installed, will not use colors when logging. To enable colors, please install the 'colored' module: python3 -m pip install colored
[08/05/2025-12:37:27] [TRT] [I] Loaded engine size: 0 MiB
[08/05/2025-12:37:27] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +0, now: CPU 0, GPU 0 (MiB)


tensor([[-0.3619,  0.2808],
        [-0.3762,  0.3632],
        [-0.4330,  0.3845]], device='cuda:0', grad_fn=<AddmmBackward0>)

Let's see history of what happend under the hood.

In [10]:
PatchedModule.history

['New top module: SimpleNet 📊97 level=0🪜 state=init⏳ call_count=1',
 'New child module: Linear 📊55 level=1🪜 state=init⏳ call_count=1',
 'New child module: Linear 📊30 level=1🪜 state=init⏳ call_count=1',
 'New child module: Linear 📊12 level=1🪜 state=init⏳ call_count=1',
 'No graph breaks in SimpleNet 📊97. Checking took 1.74s',
 'Tuning SimpleNet 📊97 took 6.59s',
 'Unpatching child module: Linear 📊12 level=1🪜 state=detached☑️ call_count=2',
 'Unpatching child module: Linear 📊30 level=1🪜 state=detached☑️ call_count=2',
 'Unpatching child module: Linear 📊55 level=1🪜 state=detached☑️ call_count=2',
 'Model tuned: SimpleNet 📊97 level=0🪜 state=tuned🎯 (TensorRTBackend) call_count=2']

Let's see final hierarchy of a tuned model.

In [11]:
PatchedModule.print_hierarchy()

PatchedModule Hierarchy:
├─ SimpleNet 📊97 level=0🪜 state=tuned🎯 (TensorRTBackend) call_count=2
  ├─ Linear 📊55 level=1🪜 state=detached☑️ call_count=2
  ├─ Linear 📊30 level=1🪜 state=detached☑️ call_count=2
  ├─ Linear 📊12 level=1🪜 state=detached☑️ call_count=2
